# Cosmogenic muons in King CRAB: Nexus photons to PMT records

This notebook treats the exported Nexus tables as an experimental data set. It first reports what Nexus actually produced—including zero-photon events—before applying any PMT assumptions. It then makes a forward prediction:

`score-plane photon → photoelectron (PDE) → measured PMT impulse shape × SPE charge → measured pre-signal noise → PMT Processing cuts`.

The field-on and field-off runs contain the same 1,000 generated primaries. Comparisons are therefore paired by `event_id`. A score-plane photon is **not** automatically a photoelectron, and a photoelectron is **not** automatically an accepted oscilloscope trigger.


In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

from SRC.KingCRAB.cosmogenic import clean_waveform, maximum_in_window, split_segments, bootstrap_noise, detect_and_cluster, make_record, reconstruct, configure_waveform_model
from DATA.crab_config import CRABRunConfig
from SRC.KingCRAB.raytrace import emission_wavelength_nm
RUN_CONFIG = CRABRunConfig(gas='argon', pressure_bar=10.0)
from SRC.KingCRAB.digitized import load_digitized_datasets, log_interpolate, log_linear_extrapolation, as_plot_datasets as load_webplotdigitizer_json

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.stats import spearmanr
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
ROOT = PROJECT_ROOT
if ROOT.name == 'CODE': ROOT = ROOT.parent
DATA_ROOT = ROOT/'DATA'
DATA = DATA_ROOT/'cosmogenics'

MUON_RATE_HZ = 49.217             # bare-sky rate through the generator target
E_CHARGE_C = 1.602176634e-19
ARGON_WAVELENGTH_NM = emission_wavelength_nm(RUN_CONFIG.gas, RUN_CONFIG.pressure_bar)
PMT_VOLTAGE_V = 900.0
N_LOW_WAVELENGTH_POINTS = 8




spectral=load_digitized_datasets(DATA_ROOT/'R7378A_Spectral_Response.json')
characteristics=load_digitized_datasets(DATA_ROOT/'R7378A_Characteristics.json')
qe_128_percent=log_linear_extrapolation(spectral['Quantum Efficiency'],ARGON_WAVELENGTH_NM,N_LOW_WAVELENGTH_POINTS)
radiant_128_mA_W=log_linear_extrapolation(spectral['Cathode Radiant Sensitivity'],ARGON_WAVELENGTH_NM,N_LOW_WAVELENGTH_POINTS)
qe_from_radiant_percent=124.0*radiant_128_mA_W/ARGON_WAVELENGTH_NM
PDE_RANGE=tuple(sorted((qe_128_percent/100.0,qe_from_radiant_percent/100.0)))
PDE=float(np.sqrt(PDE_RANGE[0]*PDE_RANGE[1]))

# Existing Characterising_PMT/Ba133 convention for the digitized axes.
gain_trace=characteristics['Gain'].copy(); gain_trace[:,1]*=1.0e13
dark_current_trace=characteristics['Anode Dark Current'].copy(); dark_current_trace[:,1]*=1.0e-12
GAIN_900V=log_interpolate(gain_trace,PMT_VOLTAGE_V)
DARK_CURRENT_900V_A=log_interpolate(dark_current_trace,PMT_VOLTAGE_V)
Q_SPE_C = GAIN_900V*E_CHARGE_C
R_LOAD_OHM = 50.0
SPE_RELATIVE_SIGMA = 0.35
PRETRIGGER_NS = 20.0
TRIGGER_PEAK_NS = 43.0
WINDOW_NS = 100.0
CLUSTER_GAP_NS = 100.0
SEED = 410000

print(f'DATA gain at {PMT_VOLTAGE_V:.0f} V: {GAIN_900V:,.0f}')
print(f'DATA anode dark current at {PMT_VOLTAGE_V:.0f} V: {DARK_CURRENT_900V_A*1e9:.3f} nA')
print(f'DATA QE extrapolation at 128 nm: {qe_128_percent:.3f}%')
print(f'DATA cathode radiant sensitivity extrapolation at 128 nm: {radiant_128_mA_W:.3f} mA/W')
print(f'Central DATA-derived PDE: {100*PDE:.3f}% (range {100*PDE_RANGE[0]:.3f}–{100*PDE_RANGE[1]:.3f}%)')

events = pd.read_csv(DATA/'paired_event_summary.csv')
photons = pd.read_csv(DATA/'photon_arrivals.csv')
config = pd.read_csv(DATA/'run_configuration.csv')
assert events.event_id.is_unique and events.primary_matched.all()
event_ids = events.event_id.to_numpy()

counts = (photons.groupby(['event_id','condition']).size().unstack(fill_value=0)
          .reindex(event_ids, fill_value=0).reindex(columns=['field_off','field_on'], fill_value=0))
origins = (photons.groupby(['condition','origin']).size().unstack(fill_value=0)
           .reindex(['field_off','field_on'], fill_value=0))
exposure_s = len(events)/MUON_RATE_HZ
print(f'Loaded {len(events):,} paired muons, {len(photons):,} score-plane photons, '
      f'and {exposure_s:.3f} s bare-sky-equivalent exposure.')

## 1. Data integrity and run-level answer

The zeros are part of the measurement. Means, occupancies, and rates below use all 1,000 muons rather than only photon-positive events. The optical rate is the probability of at least one photoelectron per muon multiplied by the generator's bare-sky crossing rate; it is not yet a thresholded DAQ rate.


In [ ]:
summary_rows=[]
for condition in ['field_off','field_on']:
    n=counts[condition].to_numpy()
    n_s1=int((photons.condition.eq(condition)&photons.origin.eq('S1')).sum())
    n_s2=int((photons.condition.eq(condition)&photons.origin.eq('S2')).sum())
    p_hit=np.mean(1-(1-PDE)**n)
    summary_rows.append({
        'condition':condition, 'muons':len(n), 'S1 photons':n_s1, 'S2 photons':n_s2,
        'all photons':int(n.sum()), 'zero-photon fraction':np.mean(n==0),
        'photons/muon':n.mean(), 'median photons/muon':np.median(n),
        '99th percentile':np.percentile(n,99), 'maximum':n.max(),
        'expected PE/muon':PDE*n.mean(), 'P(>=1 PE)/muon':p_hit,
        'optical occupancy rate [Hz]':MUON_RATE_HZ*p_hit})
run_summary=pd.DataFrame(summary_rows).set_index('condition')
display(run_summary.round(5))

fig,ax=plt.subplots(1,3,figsize=(15,4.2))
bins=np.arange(-.5,counts.field_on.max()+1.5)
for c,color in [('field_off','tab:orange'),('field_on','tab:blue')]:
    ax[0].hist(counts[c],bins=bins,histtype='step',lw=2,label=c.replace('_',' '),color=color)
ax[0].set(yscale='log',xlabel='score-plane photons per muon',ylabel='events',title='All events, including zeros'); ax[0].legend()
ax[1].scatter(counts.field_off,counts.field_on,s=18,alpha=.55)
ax[1].set(xlabel='field-off photons',ylabel='field-on photons',title='Paired event comparison')
delta=counts.field_on-counts.field_off
ax[2].hist(delta,bins=40,histtype='stepfilled',alpha=.35,color='tab:purple')
ax[2].axvline(0,color='k',ls='--'); ax[2].set(xlabel='field-on − field-off photons',ylabel='events',title='Paired field effect')
fig.tight_layout(); plt.show()
print(f'Field-on/off photon ratio: {counts.field_on.sum()/max(counts.field_off.sum(),1):.1f}.')
print(f'Events with more photons field-on: {(delta>0).sum()}; equal: {(delta==0).sum()}; fewer: {(delta<0).sum()}.')

## 2. What makes the photons

The field-on excess should be identified as S2 before it is called a PMT effect. Absolute Nexus arrival times are retained. Short-time clustering is also measured directly because photons separated by hundreds of microseconds do not form one large PMT pulse.


In [ ]:
cluster_rows=[]
for c in ['field_off','field_on']:
    grouped=photons.query('condition == @c').groupby('event_id').arrival_time_ns
    maxima=pd.Series({i:maximum_in_window(g,10.0) for i,g in grouped}).reindex(event_ids,fill_value=0)
    cluster_rows.append({'condition':c,'median max photons/10 ns':maxima.median(),
                         '90%':maxima.quantile(.9),'99%':maxima.quantile(.99),'maximum':maxima.max()})
display(pd.DataFrame(cluster_rows).set_index('condition'))

fig,ax=plt.subplots(1,3,figsize=(15,4.2))
origins.plot.bar(ax=ax[0],color=['tab:green','tab:purple']); ax[0].set(yscale='log',ylabel='score-plane photons',title='Photon origin')
for origin,g in photons.query("condition == 'field_on'").groupby('origin'):
    ax[1].hist(g.arrival_time_ns/1e3,bins=100,histtype='step',lw=1.8,label=origin)
ax[1].set(xlabel='absolute arrival time [µs]',ylabel='photons',title='Field-on timing'); ax[1].legend()
ax[2].scatter(events.deposited_energy_field_on_MeV,counts.field_on,s=16,alpha=.5)
rho,p=spearmanr(events.deposited_energy_field_on_MeV,counts.field_on)
ax[2].set(xlabel='field-on deposited energy [MeV]',ylabel='field-on photons',title=f'Energy relation: Spearman ρ={rho:.2f}')
fig.tight_layout(); plt.show()

dep=pd.DataFrame({'off':events.deposited_energy_field_off_MeV,'on':events.deposited_energy_field_on_MeV})
print('Deposited-energy paired correlation:',dep.corr().iloc[0,1].round(3))
print('Photon timing range:',photons.arrival_time_ns.min()/1e3,'to',photons.arrival_time_ns.max()/1e3,'µs')

## 3. Measured PMT response and measured baseline noise

The illuminated `Without Window/900` traces provide a **shape**, not a one-photoelectron amplitude. Their aligned average is normalized to unit charge. The amplitude comes from the 900 V datasheet gain, $Q_{1\rm PE}=Ge$. Only samples in the pre-signal interval of real traces enter the noise bank. Every synthetic waveform is filled by block-bootstrap resampling from that bank—there is no fitted Gaussian noise and no added dark-current pulse population.


In [ ]:
PMT_DIR=Path('/Volumes/Untitled/Without Window/900')
if not PMT_DIR.exists():
    raise FileNotFoundError('Mount the PMT data volume: '+str(PMT_DIR))


traces=[]
for path in sorted(PMT_DIR.glob('C1C*.txt'))[:10]:
    x=np.loadtxt(path,delimiter=',',skiprows=504,usecols=(0,1))
    traces.extend(split_segments(x))
print(f'Loaded {len(traces):,} measured illuminated PMT records from 10 files.')

n=min(map(len,traces)); raw_t=(traces[0][:n,0]-traces[0][0,0])*1e9
raw=np.vstack([x[:n,1] for x in traces])
pre=raw_t<PRETRIGGER_NS
corrected=raw-raw[:,pre].mean(axis=1,keepdims=True)
pre_rms=corrected[:,pre].std(axis=1)
med=np.median(pre_rms)
good=(pre_rms>.5*med)&(pre_rms<1.5*med)&(np.max(np.abs(corrected[:,pre]),axis=1)<4*med)
noise_bank=[row[pre].copy() for row in corrected[good]]

average=corrected.mean(axis=0)
peak=np.argmin(average); left=peak; right=peak
while left>0 and average[left]<0: left-=1
while right<n-1 and average[right]<0: right+=1
kernel_t=raw_t[left:right+1]-raw_t[left]
kernel=np.maximum(-average[left:right+1],0)
kernel/=trapezoid(kernel,kernel_t*1e-9)
kernel_peak_offset=kernel_t[np.argmax(kernel)]
DT_NS=float(np.median(np.diff(raw_t)))


t_test=np.arange(0,WINDOW_NS,DT_NS); rng=np.random.default_rng(SEED)
test=np.vstack([bootstrap_noise(rng,len(t_test)) for _ in range(1000)])
fig,ax=plt.subplots(1,3,figsize=(15,4))
for row in corrected[:40]: ax[0].plot(raw_t,row*1e3,color='0.4',alpha=.12,lw=.6)
ax[0].plot(raw_t,average*1e3,color='tab:red',lw=2,label='aligned average')
ax[0].axvspan(0,PRETRIGGER_NS,color='tab:blue',alpha=.12,label='noise source only'); ax[0].legend()
ax[0].set(xlabel='record time [ns]',ylabel='voltage [mV]',title='Measured illuminated records')
ax[1].hist(pre_rms[good]*1e3,bins=40,histtype='step',lw=2,label='measured pre-signal')
ax[1].hist(test.std(axis=1)*1e3,bins=40,histtype='step',lw=2,label='synthetic whole record')
ax[1].set(xlabel='RMS [mV]',ylabel='records',title='Noise closure'); ax[1].legend()
ax[2].plot(kernel_t,kernel/kernel.max(),lw=2)
ax[2].set(xlabel='time after impulse [ns]',ylabel='normalized amplitude',title='Empirical PMT impulse shape')
fig.tight_layout(); plt.show()
print(f'Median individual pre-signal RMS = {med*1e3:.3f} mV')
print(f'Datasheet SPE charge = {Q_SPE_C:.3e} C at gain {GAIN_900V:,.0f}')
print('The measured light-pulse amplitude is deliberately not used as one PE.')

## 4. Predictive photon-to-record model

Each score-plane photon is independently accepted with the stated PDE. Accepted photons are grouped only when separated by at most 100 ns; this prevents a millisecond-long S2 track from being collapsed into one artificial giant pulse. Each PE receives a positive gamma-distributed charge with 35% relative width. The measured impulse is scaled so its voltage area across 50 Ω equals that charge.

Reconstruction follows the laboratory notebooks: mean pre-signal baseline, pre-signal RMS, negative pulse height, a 5σ height threshold, and charge integrated over the contiguous negative pulse around the minimum. A noise-only external-trigger record is retained when an event produces no PE, so field-off is shown honestly rather than conditioned on the rare success.


In [ ]:
configure_waveform_model(PDE=PDE, CLUSTER_GAP_NS=CLUSTER_GAP_NS, WINDOW_NS=WINDOW_NS, DT_NS=DT_NS, PRETRIGGER_NS=PRETRIGGER_NS, noise_bank=noise_bank, SEED=SEED, TRIGGER_PEAK_NS=TRIGGER_PEAK_NS, kernel_peak_offset=kernel_peak_offset, kernel_t=kernel_t, kernel=kernel, SPE_RELATIVE_SIGMA=SPE_RELATIVE_SIGMA, Q_SPE_C=Q_SPE_C, R_LOAD_OHM=R_LOAD_OHM)
grouped={(c,int(i)):g for (c,i),g in photons.groupby(['condition','event_id'])}
records=[]; examples={}
for ci,c in enumerate(['field_off','field_on']):
    for eid in event_ids:
        rng=np.random.default_rng(SEED+100000*ci+int(eid))
        rows=grouped.get((c,int(eid)),photons.iloc[:0])
        clusters=detect_and_cluster(rows,rng)
        # Display/reconstruct the strongest cluster in the event.  Using the
        # first cluster would hide a later S2 burst and misrepresent its timing.
        chosen=max(clusters,key=len) if clusters else []
        t,v,s,peaks=make_record(chosen,rng); rec=reconstruct(t,v)
        records.append({'condition':c,'event_id':eid,'incident_photons':len(rows),
                        'detected_pe':sum(map(len,clusters)),'n_clusters':len(clusters),
                        'max_cluster_pe':max(map(len,clusters),default=0),
                        **{k:v for k,v in rec.items() if k not in ['left','right','minimum']}})
        examples[(c,int(eid))]=(t,v,s,peaks,rec,clusters)
reco=pd.DataFrame(records)

detector_summary=reco.groupby('condition').agg(
    events=('event_id','size'),incident_photons=('incident_photons','sum'),detected_PE=('detected_pe','sum'),
    events_with_PE=('detected_pe',lambda x:(x>0).sum()),PE_clusters=('n_clusters','sum'),
    height_pass=('height_pass','sum'),median_noise_mV=('noise_rms_V',lambda x:1e3*x.median()),
    median_height_mV=('height_V',lambda x:1e3*x.median()),median_charge_pC=('charge_C',lambda x:1e12*x.median()))
detector_summary['height-pass rate [Hz]']=detector_summary.height_pass/len(events)*MUON_RATE_HZ
display(detector_summary.round(4))

## 5. Field-on and field-off trigger records

The panels below use the same event ID in both configurations. Orange denotes field-off and blue denotes field-on. Solid vertical lines mark accepted PE arrival peaks, the red dashed line is the event's 5σ threshold, and the shaded region is the reconstructed charge interval. “NO PE” is an intentional noise-only prediction, not a missing plot.


In [ ]:
# Prefer a paired event with a detected field-on cluster; show event 0 when available.
candidate=reco.query("condition=='field_on' and max_cluster_pe>0").sort_values(
    ['max_cluster_pe','detected_pe','incident_photons'],ascending=False)
show_ids=[0]+[int(x) for x in candidate.event_id if int(x)!=0][:2]
fig,axes=plt.subplots(len(show_ids),2,figsize=(13,3.25*len(show_ids)),sharex=True)
colors={'field_off':'tab:orange','field_on':'tab:blue'}
for row,eid in enumerate(show_ids):
    for col,c in enumerate(['field_off','field_on']):
        ax=axes[row,col]; t,v,s,peaks,rec,clusters=examples[(c,eid)]
        y=(v-rec['baseline_V'])*1e3
        ax.plot(t,y,color=colors[c],lw=.9,label='synthetic record')
        ax.plot(t,s*1e3,color='k',lw=1.3,alpha=.75,label='true PMT signal')
        ax.axhline(-rec['threshold_V']*1e3,color='tab:red',ls='--',label='−5σ trigger level')
        ax.axvspan(t[rec['left']],t[rec['right']],color='tab:green',alpha=.12,label='charge interval')
        for k,(tp,origin,q) in enumerate(peaks):
            ax.axvline(tp,color='tab:purple' if origin=='S2' else 'tab:green',ls='-',alpha=.8,
                       label=f'{origin} PE peak' if k==0 else None)
        state='TRIGGER' if rec['height_pass'] else 'NO TRIGGER'
        ax.set_title(f'event {eid} — {c.replace("_"," ")} — {len(peaks)} PE in record — {state}')
        ax.set_ylabel('baseline-subtracted voltage [mV]')
        ax.text(.02,.06,f'incident={len(grouped.get((c,eid),[]))}, detected/event={sum(map(len,clusters))}\n'
                f'H={rec["height_V"]*1e3:.2f} mV, 5σ={rec["threshold_V"]*1e3:.2f} mV',
                transform=ax.transAxes,fontsize=8,bbox=dict(fc='white',alpha=.8,ec='none'))
axes[-1,0].set_xlabel('local trigger-record time [ns]'); axes[-1,1].set_xlabel('local trigger-record time [ns]')
axes[0,0].legend(fontsize=7,loc='upper right'); fig.tight_layout()
plt.show()

### Transparency-style aligned average

The laboratory transparency plots become clean because many synchronized records are averaged. The fair analogue here averages the same 1,000 external-muon-trigger records in each field configuration, including noise-only events. No record is selected for having a pulse. The colored trace is signal plus measured-baseline noise; black is the hidden forward-model signal used only as a closure check. The trigger decision is then recomputed on the averaged trace using its own pre-signal noise.


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4.3),sharey=True)
average_rows=[]
for j,(c,color) in enumerate([('field_off','tab:orange'),('field_on','tab:blue')]):
    stack=np.vstack([examples[(c,int(eid))][1] for eid in event_ids])
    truth=np.vstack([examples[(c,int(eid))][2] for eid in event_ids])
    avg=stack.mean(axis=0); avg_truth=truth.mean(axis=0); avgrec=reconstruct(t,avg)
    y=(avg-avgrec['baseline_V'])*1e3
    ax[j].plot(t,y,color=color,lw=1.5,label=f'average of {len(stack):,} muon gates')
    ax[j].plot(t,avg_truth*1e3,color='k',lw=2,label='expected PMT signal')
    ax[j].axvline(TRIGGER_PEAK_NS,color='tab:purple',ls=':',lw=1.5,label='aligned first-PE peak')
    ax[j].axhline(-avgrec['threshold_V']*1e3,color='tab:red',ls='--',label='−5σ of averaged baseline')
    state='TRIGGER' if avgrec['height_pass'] else 'NO TRIGGER'
    ax[j].set(title=f'{c.replace("_"," ")} aligned average — {state}',xlabel='record time [ns]')
    ax[j].text(.02,.06,f'pre-signal RMS={avgrec["noise_rms_V"]*1e3:.3f} mV\n'
               f'height={avgrec["height_V"]*1e3:.3f} mV',transform=ax[j].transAxes,
               bbox=dict(fc='white',alpha=.85,ec='none'))
    average_rows.append({'condition':c,'average_noise_mV':avgrec['noise_rms_V']*1e3,
                         'average_height_mV':avgrec['height_V']*1e3,
                         'average_5sigma_mV':avgrec['threshold_V']*1e3,
                         'average_trigger':avgrec['height_pass']})
ax[0].set_ylabel('average baseline-subtracted voltage [mV]'); ax[0].legend(fontsize=8)
fig.tight_layout(); plt.show()
display(pd.DataFrame(average_rows).set_index('condition').round(4))

## 6. Complete paired comparison and uncertainty from the VUV efficiency

The stochastic realization above is useful for waveform appearance, but the expected occupancy can be calculated exactly from each event's photon count. This avoids drawing a physics conclusion from one Monte Carlo seed. The low/central/high curves represent only the extrapolated VUV efficiency range; geometry, window transmission, collection efficiency, PMT gain spread, and DAQ threshold remain separate systematic uncertainties.


In [ ]:
pde_grid=np.linspace(.005,.10,160)
fig,ax=plt.subplots(1,3,figsize=(15,4.2))
for c,color in [('field_off','tab:orange'),('field_on','tab:blue')]:
    n=counts[c].to_numpy()
    occupancy=np.array([np.mean(1-(1-p)**n) for p in pde_grid])
    ax[0].plot(100*pde_grid,occupancy,color=color,lw=2,label=c.replace('_',' '))
    ax[1].plot(100*pde_grid,MUON_RATE_HZ*occupancy,color=color,lw=2,label=c.replace('_',' '))
for a in ax[:2]:
    a.axvspan(100*PDE_RANGE[0],100*PDE_RANGE[1],color='0.5',alpha=.15,label='VUV extrapolation band')
ax[0].set(xlabel='photon detection efficiency [%]',ylabel='P(at least one PE per muon)',title='Exact event-level occupancy')
ax[1].set(xlabel='photon detection efficiency [%]',ylabel='bare-sky optical occupancy rate [Hz]',title='Expected rate before threshold')

paired=reco.pivot(index='event_id',columns='condition',values=['detected_pe','height_V','charge_C','height_pass'])
ax[2].hist((paired.detected_pe.field_on-paired.detected_pe.field_off),bins=np.arange(-1.5,15.5),
           histtype='step',lw=2,color='tab:purple')
ax[2].set(xlabel='sampled detected PE: field-on − field-off',ylabel='paired events',title='One reproducible detector realization')
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8); fig.tight_layout()
plt.show()

for p in [PDE_RANGE[0],PDE,PDE_RANGE[1]]:
    line=[]
    for c in ['field_off','field_on']:
        n=counts[c].to_numpy(); occ=np.mean(1-(1-p)**n)
        line.append(f'{c}: {MUON_RATE_HZ*occ:.3f} Hz')
    print(f'PDE={100*p:.3f}% -> '+', '.join(line))

## 7. Interpretation and limits

1. **Nexus result:** field-off is nearly optically silent at the scoring plane (19 photons in 1,000 muons). Field-on produces a large S2 population, but its photons are distributed across as much as about 0.84 ms.
2. **PMT consequence:** at the central DATA-derived efficiency the field-on mean is below one PE per muon. Most individual PMT records should therefore be noise-only or small-SPE pulses; only the rare tight photon clusters produce conspicuously larger pulses. Making every event look like the measured transparency average would contradict the Nexus counts.
3. **What the transparency data establish:** the observed impulse timing/shape and the pre-signal noise. The synchronized transparency pulse is multi-PE and cannot set the SPE gain without a calibrated incident-photon count.
4. **Field discriminator:** the strongest robust separation is occupancy/rate and S2 timing, not merely average waveform height. A real comparison should externally trigger on a muon, retain the long drift gate, and compare prompt S1-like and delayed S2-like cluster counts with identical PMT cuts.
5. **Still required for an absolute prediction:** measured 128-nm PDE (including collection efficiency), optical-window transmission if installed, electronics transfer function, trigger dead time, and an SPE calibration. The notebook keeps these explicit so measurements can replace assumptions without changing the analysis.

All figures and tables remain embedded in this notebook. It deliberately does not create CSV or PNG side products.
